In [6]:
"""Figure 5: qPCR positivity rate by Poisson genome-span bin, one chart per
virus, with the logistic regression statistics in a box inside the plot.

Span metric: genome_span_poisson throughout (per-window 1 - e^(-average_coverage),
width-averaged over the genome). This figure is Poisson-only.

Chart:
  - Bars: qPCR-positive % per 0.1-wide span bin. The axis is fixed at 0-1 for every
    virus, so bins a virus never reaches are blank slots. A 0% bar still gets its
    label and n dot.
  - Right axis (log scale): n per bin, dark line with white markers.
  - Stats box inside the axes: odds ratio per +0.1 span, its 95% CI and the p-value
    (ordinary, unclustered SEs). Box position is set per virus (STATS_BOX_POS) to avoid bars.

Statistics:
  - Primary: logistic regression, qPCR detected (1/0) ~ genome_span_poisson, fitted
    on continuous span (not the bins). OR is reported per +0.1 span, with ordinary
    (unclustered) model-based standard errors, consistent with the rest of the study.
    No fitted curve is drawn: several viruses have empty bins inside their span range,
    and a curve would bridge them as if there were data there.

Outputs:
  fig5_qpcr_positivity_final_{slug}.png / .svg
  fig5_qpcr_positivity_final_stats.csv
"""

#Import required packages
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
import statsmodels.api as sm

#Set data path and statisic setups.
DATA_PATH = "Data_3_Correlation_Data_Final.xlsx"
SPAN_COL = "genome_span_poisson"
BIN_WIDTH = 0.1
MAX_SPAN = 1.0
Z = 1.96  # 95%
STATS_PATH = "fig5_qpcr_positivity_final_stats.csv"

# Stats box anchor (x, y, horizontal alignment) in axes fractions, top edge at y.
# Default is top-left; overrides move it into empty bins, or for SARS-CoV-2 (bars fill
# every bin to ~100%) onto the bar bodies below the % labels and above the n line.
STATS_BOX_DEFAULT = (0.03, 0.97, "left")
STATS_BOX_POS = {
    "rsva": (0.99, 0.62, "right"),   # below the 0.6-0.7 bar's 100% label, inside empty 0.7-1.0
    "rsvb": (0.985, 0.97, "right"),
    "sarscov2": (0.985, 0.70, "right"),
}

COLOR_N = "#333333"    # n-per-bin line on the right log axis
LABEL_FONTSIZE = 13    # % labels above bars (10 bins per chart)
TICK_FONTSIZE = 11     # x tick labels

VIRUSES = [
    ("fluA", "Influenza A", "flua", "#3C83D9"),
    ("fluB", "Influenza B", "flub", "#EC7444"),
    ("rsvA", "RSV A", "rsva", "#2CB483"),
    ("rsvB", "RSV B", "rsvb", "#5748AD"),
    ("sars-cov-2", "SARS-CoV-2 (N1)", "sarscov2", "#E45655"),
]


def load_seq_detected(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    df = df[(df["mismatch_flag_after_correction"] == "none")
            & (df["sequencing_detected"] == "detected")].copy()
    df["qpcr_pos"] = (df["qpcr_detected"] == "detected").astype(int)
    return df


def bin_rates(df):
    """qPCR-positive % and n per fixed 0.1 bin on [0, 1]; empty bins kept (rate NaN)."""
    edges = np.linspace(0, MAX_SPAN, int(round(MAX_SPAN / BIN_WIDTH)) + 1)
    b = pd.cut(df[SPAN_COL], edges, include_lowest=True)
    g = df.groupby(b, observed=False)["qpcr_pos"]
    n, k = g.size(), g.sum()
    rate = k / n.replace(0, np.nan) * 100
    labels = [f"{max(iv.left, 0):.1f}–{iv.right:.1f}" for iv in n.index]  # include_lowest nudges edge 0 below 0
    return labels, n.to_numpy(), k.to_numpy(), rate.to_numpy()


def fit_logistic(df):
    """Logistic regression with ordinary (unclustered) SEs. Returns the stats dict."""
    X = sm.add_constant(df[[SPAN_COL]])
    model = sm.Logit(df["qpcr_pos"], X).fit(disp=0)
    b, se = model.params[SPAN_COL], model.bse[SPAN_COL]
    return {
        "n": len(df), "qpcr_positive": int(df["qpcr_pos"].sum()),
        "OR_per_0.1": np.exp(BIN_WIDTH * b),
        "OR_ci_low": np.exp(BIN_WIDTH * (b - Z * se)),
        "OR_ci_high": np.exp(BIN_WIDTH * (b + Z * se)),
        "logit_p": model.pvalues[SPAN_COL],
    }


def fmt_p(p):
    if p < 0.001:
        mant, exp = f"{p:.1e}".split("e")
        return f"p = {mant}×10$^{{{int(exp)}}}$"
    return f"p = {p:.3f}"


def make_chart(df, label, color, out_path, box_pos):
    labels, n, k, rate = bin_rates(df)
    stats = fit_logistic(df)
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.bar(x, rate, color=color, edgecolor="black", alpha=0.75, linewidth=0.7, zorder=3)
    ax.set_xlim(-0.5, len(labels) - 0.5)

    # n per bin on a log right axis; empty bins get no dot
    ax_n = ax.twinx()
    ax_n.plot(x, np.where(n > 0, n, np.nan), color=COLOR_N, linewidth=2, marker="o", markersize=8,
              markerfacecolor="white", markeredgewidth=2, zorder=4)
    ax_n.set_yscale("log")
    ax_n.set_ylim(0.7, 10000)  # headroom keeps the line down among the bars
    ax_n.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, pos: f"{v:g}"))
    ax_n.set_ylabel("Samples per bin (n, log scale)", fontsize=18, color=COLOR_N)
    ax_n.tick_params(axis="y", labelsize=14, colors=COLOR_N)
    ax_n.spines["top"].set_visible(False)

    # % labels on the top-most axis (in left-axis data coords) with a white backing,
    # so the n line never hides them
    for xi, r in zip(x, rate):
        if np.isnan(r):
            continue
        ax_n.text(xi, r + 1.5, "100%" if r == 100 else f"{r:.1f}%", transform=ax.transData,
                  ha="center", va="bottom", fontsize=LABEL_FONTSIZE, color="#333333", zorder=6,
                  bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Genome Span Bin", fontsize=18)
    ax.set_ylabel("qPCR-positive rate (%)", fontsize=18)
    ax.set_ylim(0, 120)                    # headroom above 100% for the bar labels
    ax.set_yticks(np.arange(0, 101, 20))   # a rate cannot exceed 100%, so no tick above it
    ax.set_title(f"{label}: qPCR Positivity Rate by Genome Span (Poisson) Bin\n(Among Sequencing-Detected Samples)",
                 fontsize=18, fontweight="bold", loc="center", pad=30)
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="x", labelsize=TICK_FONTSIZE)
    ax.spines["top"].set_visible(False)
    ax.grid(axis="y", which="major", color="#e1e0d9", linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

    # Stats box inside the axes (on the top-most axis so the n line can't cover it)
    box = (f"Logistic regression\n"
           f"OR = {stats['OR_per_0.1']:.2f} per +0.1 span\n"
           f"95% CI: {stats['OR_ci_low']:.2f}–{stats['OR_ci_high']:.2f}\n"
           f"{fmt_p(stats['logit_p'])}")
    bx, by, ha = box_pos
    ax_n.text(bx, by, box, transform=ax.transAxes, ha=ha, va="top", multialignment="left", fontsize=12, color="#222222",
              linespacing=1.5, zorder=7,
              bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#888888", linewidth=1))

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.25)
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"[{label}] n={stats['n']} OR={stats['OR_per_0.1']:.2f} "
          f"({stats['OR_ci_low']:.2f}-{stats['OR_ci_high']:.2f}) p={stats['logit_p']:.2g} -> {out_path}")
    return stats


def main():
    rows = []
    for sheet, label, slug, color in VIRUSES:
        stats = make_chart(load_seq_detected(sheet), label, color, f"fig5_qpcr_positivity_final_{slug}.png",
                           STATS_BOX_POS.get(slug, STATS_BOX_DEFAULT))
        rows.append({"virus": label, **stats})
    pd.DataFrame(rows).to_csv(STATS_PATH, index=False)
    print(f"stats -> {STATS_PATH}")


if __name__ == "__main__":
    main()


[Influenza A] n=1400 OR=1.51 (1.31-1.73) p=6.2e-09 -> fig5_qpcr_positivity_final_flua.png
[Influenza B] n=663 OR=2.61 (1.53-4.46) p=0.00045 -> fig5_qpcr_positivity_final_flub.png
[RSV A] n=963 OR=2.70 (2.12-3.43) p=4.9e-16 -> fig5_qpcr_positivity_final_rsva.png
[RSV B] n=347 OR=3.43 (1.95-6.05) p=1.9e-05 -> fig5_qpcr_positivity_final_rsvb.png
[SARS-CoV-2 (N1)] n=3209 OR=5.08 (3.40-7.59) p=2.3e-15 -> fig5_qpcr_positivity_final_sarscov2.png
stats -> fig5_qpcr_positivity_final_stats.csv


In [1]:
"""
Supp: Figure 3:genome_span_poisson vs qPCR detect/non-detect rate of all viruses

Same population, same fixed 0.1-wide genome_span bins, same per-virus
color scheme. Each bin contributes two mirrored grouped bars per virus:
  - above zero: qPCR-detect rate (%)
  - below zero: -(qPCR-non-detect rate) = -(100 - detect rate)
  
Detect rate + non-detect rate = 100% exactly (every sequencing-detected
sample in Data_3 is either qPCR-detected or not, no "not tested" category),
so the below-zero bar is a mathematical mirror of the above-zero bar, not an
independently-fit value -- as genome_span increases and detect rate rises
toward 100%, the non-detect bar shrinks toward zero in lockstep.

All 5 viruses share one fixed global bin grid (0 to the largest max-genome-
span across viruses) so bars are dodged consistently; a virus with no
sequencing-detected samples in a given bin simply has no bar there.

Outputs:
  fig5_qpcr_positivity_final_{slug}.png / .svg
  fig5_qpcr_positivity_final_stats.csv
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

DATA_PATH = "Data_3_Correlation_Data_Final.xlsx"
BIN_WIDTH = 0.1
SPAN_COL = "genome_span_poisson"

VIRUSES = [
    ("fluA", "Influenza A", "#3C83D9"),
    ("fluB", "Influenza B", "#EC7444"),
    ("rsvA", "RSV A", "#2CB483"),
    ("rsvB", "RSV B", "#5748AD"),
    ("sars-cov-2", "SARS-CoV-2 (N1)", "#E45655"),
]


def load_sheet(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    return df[df["mismatch_flag_after_correction"] == "none"].copy()


def compute_rate_on_grid(df, bin_edges):
    seq_pos = df[df["sequencing_detected"] == "detected"].copy()
    seq_pos["bin"] = pd.cut(seq_pos[SPAN_COL], bin_edges, include_lowest=True)

    grouped = seq_pos.groupby("bin", observed=False)
    detected = grouped["qpcr_detected"].apply(lambda s: (s == "detected").sum())
    n = grouped.size()
    rate = (detected / n * 100)
    return rate.to_numpy(), n.to_numpy()


def main():
    per_virus = {}
    global_max_span = 0.0
    for sheet, label, color in VIRUSES:
        df = load_sheet(sheet)
        seq_pos = df[df["sequencing_detected"] == "detected"]
        global_max_span = max(global_max_span, seq_pos[SPAN_COL].max())
        per_virus[sheet] = df

    max_edge = np.ceil(global_max_span / BIN_WIDTH) * BIN_WIDTH
    bin_edges = np.arange(0, max_edge + BIN_WIDTH, BIN_WIDTH)
    n_bins = len(bin_edges) - 1
    bin_labels = [f"{bin_edges[i]:.1f}–{bin_edges[i+1]:.1f}" for i in range(n_bins)]

    results = {}
    for sheet, label, color in VIRUSES:
        rate, n = compute_rate_on_grid(per_virus[sheet], bin_edges)
        results[sheet] = (rate, n)

    n_viruses = len(VIRUSES)
    group_width = 0.82
    bar_width = group_width / n_viruses
    x = np.arange(n_bins)

    fig, ax = plt.subplots(figsize=(12, 8))

    for bi in range(n_bins):
        if bi % 2 == 1:
            ax.axvspan(bi - 0.5, bi + 0.5, color="#e8e8e8", alpha=0.6, zorder=0)

    for i, (sheet, label, color) in enumerate(VIRUSES):
        rate, n = results[sheet]
        valid = ~np.isnan(rate)
        offset = (i - (n_viruses - 1) / 2) * bar_width
        xi = x[valid] + offset

        ax.bar(xi, rate[valid], width=bar_width, color=color, edgecolor="black",
               linewidth=0.7, alpha=0.9, zorder=3)
        ax.bar(xi, -(100 - rate[valid]), width=bar_width, color=color, edgecolor="black",
               linewidth=0.7, alpha=0.55, zorder=3)

    ax.axhline(0, color="black", linewidth=1.1, zorder=4)

    ax.set_xlabel("Genome Span Bin", fontsize=18)
    ax.set_ylabel("qPCR-positive rate (%)", fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels)
    ax.set_ylim(-100, 100)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{abs(y):.0f}"))
    ax.text(0.01, 0.97, "Detect", transform=ax.transAxes, fontsize=13,
            fontweight="bold", color="#333333", va="top")
    ax.text(0.01, 0.03, "Non-detect", transform=ax.transAxes, fontsize=13,
            fontweight="bold", color="#333333", va="bottom")
    ax.set_title("qPCR Detect vs. Non-Detect Rate by Genome Span, All Viruses\n"
                 "(Among Sequencing-Detected Samples; solid = detect, faded = non-detect)",
                 fontsize=17, fontweight="bold", loc="center", pad=12)
    ax.tick_params(axis="both", labelsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", which="major", color="#e1e0d9", linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

    virus_handles = [Patch(facecolor=color, edgecolor="black", linewidth=0.7, label=label)
                     for sheet, label, color in VIRUSES]
    ax.legend(handles=virus_handles, loc="lower right", fontsize=12, frameon=False,
              title="Virus", title_fontsize=12)

    out_path = "Supp_fig_3_genome_span_poisson_detect_allviruses.png"
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.25)
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"saved -> {out_path}")


if __name__ == "__main__":
    main()


saved -> Supp_fig_3_genome_span_poisson_detect_allviruses.png


In [3]:
"""Figure 5f: norovirus: qPCR positivity rate by Poisson genome-span bin,
with the logistic regression statistics in a box inside the plot. 

Norovirus was added to assess if year-round persistence in wastewater played a role in qPCR and sequencing detections.

Chart:
  - Bars: qPCR-positive % per 0.1-wide span bin, fixed 0-1 axis.
  - Right axis (log scale): n per bin, dark line with white markers.
  - Stats box inside the axes: odds ratio per +0.1 span, its 95% CI and the p-value
    (ordinary, unclustered SEs).

Statistics:
  - Logistic regression, qPCR detected (1/0) ~ genome_span_poisson, fitted on
    continuous span (not the bins). OR reported per +0.1 span, unclustered SEs.

Outputs (this folder):
  fig5f_qpcr_positivity_final_norovirus.png / .svg
  fig5f_qpcr_positivity_final_stats_norovirus.csv
"""

#Import required packages
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
import statsmodels.api as sm

#Path to input data and statistic setup
DATA_PATH = "Data_3_Correlation_Data_Norovirus.xlsx"
SHEET = "Both"
SPAN_COL = "genome_span_poisson"
BIN_WIDTH = 0.1
MAX_SPAN = 1.0
Z = 1.96  # 95%
STATS_PATH = "fig5f_qpcr_positivity_final_stats_norovirus.csv"
OUT_PATH = "fig5f_qpcr_positivity_final_norovirus.png"

LABEL = "Norovirus"
COLOR = "#C9922E"

# Stats box anchor (x, y, horizontal alignment) in axes fractions, top edge at y.
# Top-left: norovirus bars are short in the low-span bins.
STATS_BOX = (0.03, 0.97, "left")

COLOR_N = "#333333"    # n-per-bin line on the right log axis
LABEL_FONTSIZE = 13    # % labels above bars (10 bins per chart)
TICK_FONTSIZE = 11     # x tick labels


def load_seq_detected():
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
    df = df[(df["mismatch_flag_after_correction"] == "none")
            & (df["sequencing_detected"] == "detected")].copy()
    df["qpcr_pos"] = (df["qpcr_detected"] == "detected").astype(int)
    return df


def bin_rates(df):
    """qPCR-positive % and n per fixed 0.1 bin on [0, 1]; empty bins kept (rate NaN)."""
    edges = np.linspace(0, MAX_SPAN, int(round(MAX_SPAN / BIN_WIDTH)) + 1)
    b = pd.cut(df[SPAN_COL], edges, include_lowest=True)
    g = df.groupby(b, observed=False)["qpcr_pos"]
    n, k = g.size(), g.sum()
    rate = k / n.replace(0, np.nan) * 100
    labels = [f"{max(iv.left, 0):.1f}–{iv.right:.1f}" for iv in n.index]  # include_lowest nudges edge 0 below 0
    return labels, n.to_numpy(), k.to_numpy(), rate.to_numpy()


def fit_logistic(df):
    """Logistic regression with ordinary (unclustered) SEs. Returns the stats dict."""
    X = sm.add_constant(df[[SPAN_COL]])
    model = sm.Logit(df["qpcr_pos"], X).fit(disp=0)
    b, se = model.params[SPAN_COL], model.bse[SPAN_COL]
    return {
        "n": len(df), "qpcr_positive": int(df["qpcr_pos"].sum()),
        "OR_per_0.1": np.exp(BIN_WIDTH * b),
        "OR_ci_low": np.exp(BIN_WIDTH * (b - Z * se)),
        "OR_ci_high": np.exp(BIN_WIDTH * (b + Z * se)),
        "logit_p": model.pvalues[SPAN_COL],
    }


def fmt_p(p):
    if p < 0.001:
        mant, exp = f"{p:.1e}".split("e")
        return f"p = {mant}×10$^{{{int(exp)}}}$"
    return f"p = {p:.3f}"


def make_chart(df, label, color, out_path, box_pos):
    labels, n, k, rate = bin_rates(df)
    stats = fit_logistic(df)
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.bar(x, rate, color=color, edgecolor="black", alpha=0.75, linewidth=0.7, zorder=3)
    ax.set_xlim(-0.5, len(labels) - 0.5)

    # n per bin on a log right axis; empty bins get no dot
    ax_n = ax.twinx()
    ax_n.plot(x, np.where(n > 0, n, np.nan), color=COLOR_N, linewidth=2, marker="o", markersize=8,
              markerfacecolor="white", markeredgewidth=2, zorder=4)
    ax_n.set_yscale("log")
    ax_n.set_ylim(0.7, 10000)  # headroom keeps the line down among the bars
    ax_n.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, pos: f"{v:g}"))
    ax_n.set_ylabel("Samples per bin (n, log scale)", fontsize=18, color=COLOR_N)
    ax_n.tick_params(axis="y", labelsize=14, colors=COLOR_N)
    ax_n.spines["top"].set_visible(False)

    # % labels on the top-most axis (in left-axis data coords) with a white backing,
    # so the n line never hides them
    for xi, r in zip(x, rate):
        if np.isnan(r):
            continue
        ax_n.text(xi, r + 1.5, "100%" if r == 100 else f"{r:.1f}%", transform=ax.transData,
                  ha="center", va="bottom", fontsize=LABEL_FONTSIZE, color="#333333", zorder=6,
                  bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Genome Span Bin", fontsize=18)
    ax.set_ylabel("qPCR-positive rate (%)", fontsize=18)
    ax.set_ylim(0, 120)                    # headroom above 100% for the bar labels
    ax.set_yticks(np.arange(0, 101, 20))   # a rate cannot exceed 100%, so no tick above it
    ax.set_title(f"{label}: qPCR Positivity Rate by Genome Span (Poisson) Bin\n(Among Sequencing-Detected Samples)",
                 fontsize=18, fontweight="bold", loc="center", pad=30)
    ax.tick_params(axis="y", labelsize=14)
    ax.tick_params(axis="x", labelsize=TICK_FONTSIZE)
    ax.spines["top"].set_visible(False)
    ax.grid(axis="y", which="major", color="#e1e0d9", linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

    # Stats box inside the axes (on the top-most axis so the n line can't cover it)
    box = (f"Logistic regression\n"
           f"OR = {stats['OR_per_0.1']:.2f} per +0.1 span\n"
           f"95% CI: {stats['OR_ci_low']:.2f}–{stats['OR_ci_high']:.2f}\n"
           f"{fmt_p(stats['logit_p'])}")
    bx, by, ha = box_pos
    ax_n.text(bx, by, box, transform=ax.transAxes, ha=ha, va="top", multialignment="left", fontsize=12, color="#222222",
              linespacing=1.5, zorder=7,
              bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="#888888", linewidth=1))

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.25)
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"[{label}] n={stats['n']} OR={stats['OR_per_0.1']:.2f} "
          f"({stats['OR_ci_low']:.2f}-{stats['OR_ci_high']:.2f}) p={stats['logit_p']:.2g} -> {out_path}")
    return stats


def main():
    stats = make_chart(load_seq_detected(), LABEL, COLOR, OUT_PATH, STATS_BOX)
    pd.DataFrame([{"virus": LABEL, **stats}]).to_csv(STATS_PATH, index=False)
    print(f"stats -> {STATS_PATH}")


if __name__ == "__main__":
    main()

[Norovirus] n=3908 OR=1.74 (1.66-1.82) p=3e-130 -> fig5_qpcr_positivity_final_norovirus.png
stats -> fig5_qpcr_positivity_final_stats_norovirus.csv
